In [ ]:
from datasets import load_dataset

dataset = load_dataset("Wangrongsheng/ag_news")

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

### Step 1: Load the Dataset

The `datasets` library has been installed, and the `ag_news` dataset is loaded using `load_dataset`.

In [ ]:

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [ ]:

print(dataset['train'][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


### Step 2: Clean and Normalize the Text

I will create a function to lowercase the text and remove special characters, punctuation, and numbers, then apply it to the 'text' column of the dataset.

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def apply_cleaning(batch):
    batch['text'] = [clean_text(text) for text in batch['text']]
    return batch

cleaned_dataset = dataset.map(apply_cleaning, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:

print("Original text:", dataset['train'][0]['text'])
print("Cleaned text:", cleaned_dataset['train'][0]['text'])

Original text: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Cleaned text: wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again


### Step 3: Tokenize the Strings into Numbers

Now, I'll tokenize the cleaned text, converting each word into a unique integer ID using Keras' `Tokenizer`.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer


all_text = list(cleaned_dataset['train']['text']) + list(cleaned_dataset['test']['text'])


tokenizer = Tokenizer(num_words=None, oov_token="<unk>")
tokenizer.fit_on_texts(all_text)

X_train_sequences = tokenizer.texts_to_sequences(cleaned_dataset['train']['text'])
X_test_sequences = tokenizer.texts_to_sequences(cleaned_dataset['test']['text'])

In [ ]:

print("Example cleaned text:", cleaned_dataset['train'][0]['text'])
print("Example tokenized sequence:", X_train_sequences[0])

Example cleaned text: wall st bears claw back into the black reuters reuters shortsellers wall streets dwindlingband of ultracynics are seeing green again
Example tokenized sequence: [392, 324, 1538, 14755, 100, 54, 2, 816, 24, 24, 39932, 392, 1937, 51951, 5, 39933, 35, 3856, 740, 299]


### Step 4: Pad and Truncate Sequences

I will use `pad_sequences` to ensure all input sequences have a uniform length, adding zeros to shorter sequences and truncating longer ones.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences


max_sequence_length = max(len(seq) for seq in X_train_sequences + X_test_sequences)
print(f"Maximum sequence length found: {max_sequence_length}")


MAX_LEN = 100
X_train_padded = pad_sequences(X_train_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_LEN, padding='post', truncating='post')

Maximum sequence length found: 176


In [ ]:

print("Original sequence length:", len(X_train_sequences[0]))
print("Padded sequence shape:", X_train_padded[0].shape)
print("Padded sequence example:", X_train_padded[0])

Original sequence length: 20
Padded sequence shape: (100,)
Padded sequence example: [  392   324  1538 14755   100    54     2   816    24    24 39932   392
  1937 51951     5 39933    35  3856   740   299     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0]


### Step 5: Convert Labels to Categorical Vectors

I will one-hot encode the target labels to prepare them for the model's output layer.

In [ ]:
from tensorflow.keras.utils import to_categorical


y_train = cleaned_dataset['train']['label']
y_test = cleaned_dataset['test']['label']

num_classes = 4

y_train_one_hot = to_categorical(y_train, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test, num_classes=num_classes)

In [ ]:

print("Original label:", y_train[0])
print("One-hot encoded label:", y_train_one_hot[0])

Original label: 2
One-hot encoded label: [0. 0. 1. 0.]


### Step 6: Define the RNN Model Architecture

I will now define a Sequential RNN model with an Embedding layer, a SimpleRNN layer, and a Dense output layer with softmax activation.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


vocabulary_size = len(tokenizer.word_index) + 1
model = Sequential([
    Embedding(input_dim=vocabulary_size, output_dim=64, input_length=MAX_LEN),
    SimpleRNN(units=64),
    Dense(units=num_classes, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Step 7: Compile and Train the Model

I will compile the model using the Adam optimizer and categorical cross-entropy loss, then train it using the prepared training data.

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train_padded,
                    y_train_one_hot,
                    epochs=5,
                    batch_size=64,
                    validation_split=0.2)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 92s 61ms/step - accuracy: 0.7054 - loss: 0.8654 - val_accuracy: 0.6194 - val_loss: 1.0520
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 144s 62ms/step - accuracy: 0.7002 - loss: 0.8773 - val_accuracy: 0.6159 - val_loss: 1.0506
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 143s 63ms/step - accuracy: 0.6951 - loss: 0.8843 - val_accuracy: 0.6104 - val_loss: 1.0535
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 99s 66ms/step - accuracy: 0.6823 - loss: 0.9051 - val_accuracy: 0.5964 - val_loss: 1.0702
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 141s 66ms/step - accuracy: 0.6627 - loss: 0.9217 - val_accuracy: 0.5830 - val_loss: 1.0740


### Step 8: Read and Evaluate the Target Accuracy

Finally, I will evaluate the trained model on the test set to determine its accuracy on unseen data.